<a href="https://colab.research.google.com/github/acerNZ/HAL/blob/master/Markdown_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
import subprocess
import sys

print("Installing required dependencies...")
packages = [
    'pandas',
    'openpyxl',
    'python-docx',
    'reportlab',
    'markdown2',
    'google-colab'
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Dependencies installed successfully!\n")

# ============================================================================
# MAIN SCRIPT STARTS HERE
# ============================================================================

import pandas as pd
from google.colab import files
import os
import re
from pathlib import Path
from markdown2 import markdown
from docx import Document
from docx.shared import Pt, Inches
from reportlab.lib.pagesizes import letter, A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import tempfile

class FileConverter:
    """Convert markdown files to various formats (Excel, CSV, Word, PDF)"""

    def __init__(self):
        self.supported_formats = ['excel', 'csv', 'word', 'pdf']
        self.temp_dir = tempfile.gettempdir()

    def read_markdown_file(self, file_path):
        """Read markdown file"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            return content
        except Exception as e:
            raise Exception(f"Error reading file: {str(e)}")

    def extract_tables_from_markdown(self, content):
        """Extract tables from markdown content"""
        tables = []
        lines = content.split('\n')

        i = 0
        while i < len(lines):
            line = lines[i].strip()

            # Look for markdown table start
            if '|' in line and i + 1 < len(lines) and '|' in lines[i + 1]:
                table_lines = [line]
                separator_line = lines[i + 1].strip()
                table_lines.append(separator_line)
                i += 2

                # Collect table rows
                while i < len(lines) and '|' in lines[i]:
                    table_lines.append(lines[i].strip())
                    i += 1

                # Parse table
                df = self.parse_markdown_table(table_lines)
                if df is not None:
                    tables.append(df)
                continue

            i += 1

        return tables

    def parse_markdown_table(self, table_lines):
        """Parse markdown table lines into DataFrame"""
        try:
            if len(table_lines) < 3:
                return None

            # Extract header
            header_line = table_lines[0]
            headers = [h.strip() for h in header_line.split('|')[1:-1]]

            if not headers:
                return None

            # Extract rows
            rows = []
            for line in table_lines[2:]:
                if '|' not in line:
                    continue
                cells = [c.strip() for c in line.split('|')[1:-1]]
                if len(cells) == len(headers):
                    rows.append(cells)

            if not rows:
                return None

            df = pd.DataFrame(rows, columns=headers)
            return df
        except Exception as e:
            print(f"Error parsing table: {str(e)}")
            return None

    def convert_to_excel(self, dataframes, output_path):
        """Convert to Excel format"""
        try:
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                for idx, df in enumerate(dataframes):
                    sheet_name = f'Table_{idx + 1}'
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

            print(f"✓ Excel file created: {output_path}")
            return True
        except Exception as e:
            print(f"✗ Error creating Excel file: {str(e)}")
            return False

    def convert_to_csv(self, dataframes, output_path):
        """Convert to CSV format"""
        try:
            if len(dataframes) > 1:
                # Multiple tables - save first one, warn about others
                dataframes[0].to_csv(output_path, index=False)
                print(f"⚠ Multiple tables found. Saved first table to CSV.")
                print(f"✓ CSV file created: {output_path}")
            else:
                dataframes[0].to_csv(output_path, index=False)
                print(f"✓ CSV file created: {output_path}")
            return True
        except Exception as e:
            print(f"✗ Error creating CSV file: {str(e)}")
            return False

    def convert_to_word(self, dataframes, output_path):
        """Convert to Word format"""
        try:
            doc = Document()
            doc.add_heading('Converted Data from Markdown', level=1)
            doc.add_paragraph(f'Tables converted: {len(dataframes)}')
            doc.add_paragraph()

            for idx, df in enumerate(dataframes):
                if idx > 0:
                    doc.add_page_break()

                doc.add_heading(f'Table {idx + 1}', level=2)

                # Add table to document
                table = doc.add_table(rows=1, cols=len(df.columns))
                table.style = 'Light Grid Accent 1'

                # Add header row
                header_cells = table.rows[0].cells
                for i, column in enumerate(df.columns):
                    header_cells[i].text = str(column)

                # Add data rows
                for _, row in df.iterrows():
                    row_cells = table.add_row().cells
                    for i, value in enumerate(row):
                        row_cells[i].text = str(value)

                doc.add_paragraph()

            doc.save(output_path)
            print(f"✓ Word file created: {output_path}")
            return True
        except Exception as e:
            print(f"✗ Error creating Word file: {str(e)}")
            return False

    def convert_to_pdf(self, dataframes, output_path):
        """Convert to PDF format"""
        try:
            doc = SimpleDocTemplate(output_path, pagesize=A4,
                                   rightMargin=10, leftMargin=10,
                                   topMargin=10, bottomMargin=10)
            story = []
            styles = getSampleStyleSheet()

            # Add title
            title_style = ParagraphStyle(
                'CustomTitle',
                parent=styles['Heading1'],
                fontSize=16,
                textColor=colors.HexColor('#1a1a1a'),
                spaceAfter=12
            )
            story.append(Paragraph('Converted Data from Markdown', title_style))
            story.append(Spacer(1, 0.2*inch))

            for idx, df in enumerate(dataframes):
                # Table title
                story.append(Paragraph(f'Table {idx + 1}', styles['Heading2']))
                story.append(Spacer(1, 0.1*inch))

                # Convert DataFrame to table data
                table_data = [list(df.columns)] + df.values.tolist()

                # Create table
                table = Table(table_data, repeatRows=1)
                table.setStyle(TableStyle([
                    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#4472C4')),
                    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                    ('FONTSIZE', (0, 0), (-1, 0), 9),
                    ('BOTTOMPADDING', (0, 0), (-1, 0), 8),
                    ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
                    ('GRID', (0, 0), (-1, -1), 1, colors.black),
                    ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
                    ('FONTSIZE', (0, 1), (-1, -1), 8),
                    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#F2F2F2')])
                ]))

                story.append(table)
                story.append(Spacer(1, 0.3*inch))

            doc.build(story)
            print(f"✓ PDF file created: {output_path}")
            return True
        except Exception as e:
            print(f"✗ Error creating PDF file: {str(e)}")
            return False


    def convert_file(self, file_path, output_format):
        """Main conversion method"""
        # Validate format
        if output_format.lower() not in self.supported_formats:
            print(f"✗ Format '{output_format}' not supported.")
            print(f"Supported formats: {', '.join(self.supported_formats)}")
            return False

        # Read markdown file
        print(f"\n📖 Reading markdown file...")
        content = self.read_markdown_file(file_path)

        # Extract tables
        print(f"📊 Extracting tables...")
        dataframes = self.extract_tables_from_markdown(content)

        if not dataframes:
            print("✗ No tables found in markdown file.")
            return False

        print(f"✓ Found {len(dataframes)} table(s)")

        # Generate output filename
        input_name = Path(file_path).stem
        output_format_lower = output_format.lower()

        if output_format_lower == 'excel':
            output_file = f"{input_name}_Converted.xlsx"
        elif output_format_lower == 'csv':
            output_file = f"{input_name}_Converted.csv"
        elif output_format_lower == 'word':
            output_file = f"{input_name}_Converted.docx"
        elif output_format_lower == 'pdf':
            output_file = f"{input_name}_Converted.pdf"

        output_path = os.path.join(self.temp_dir, output_file)

        # Convert based on format
        print(f"\n🔄 Converting to {output_format.upper()}...")

        if output_format_lower == 'excel':
            success = self.convert_to_excel(dataframes, output_path)
        elif output_format_lower == 'csv':
            success = self.convert_to_csv(dataframes, output_path)
        elif output_format_lower == 'word':
            success = self.convert_to_word(dataframes, output_path)
        elif output_format_lower == 'pdf':
            success = self.convert_to_pdf(dataframes, output_path)

        if success:
            print(f"\n✓ Conversion completed successfully!")
            return output_path
        else:
            return False


# ============================================================================
# INTERACTIVE USER INTERFACE
# ============================================================================

def main():
    print("=" * 60)
    print("   MARKDOWN TO MULTI-FORMAT FILE CONVERTER")
    print("   For Google Colab")
    print("=" * 60)
    print()

    converter = FileConverter()

    # Upload file
    print("📁 Uploading markdown file...")
    print("   Supported: .md (markdown) files")
    print()

    uploaded_files = files.upload()

    if not uploaded_files:
        print("✗ No file uploaded.")
        return

    file_name = list(uploaded_files.keys())[0]
    file_path = file_name

    print(f"✓ File uploaded: {file_name}")
    print()

    # Select output format
    print("Select output format:")
    print("  1. Excel (.xlsx)")
    print("  2. CSV (.csv)")
    print("  3. Word (.docx)")
    print("  4. PDF (.pdf)")
    print()

    choice = input("Enter your choice (1-4): ").strip()

    format_map = {
        '1': 'excel',
        '2': 'csv',
        '3': 'word',
        '4': 'pdf'
    }

    output_format = None
    if choice in format_map:
        output_format = format_map[choice]
    else:
        print("✗ Invalid choice. Please enter 1, 2, 3, or 4.")
        choice = input("Enter your choice again (1-4): ").strip()
        if choice in format_map:
            output_format = format_map[choice]
        else:
            print("✗ Invalid choice again. Defaulting to CSV format.")
            output_format = 'csv'


    # Convert file
    print()
    output_path = converter.convert_file(file_path, output_format)

    if output_path:
        # Download converted file
        print(f"\n📥 Downloading converted file...")
        files.download(output_path)
        print(f"✓ File ready for download!")
    else:
        print("✗ Conversion failed.")


# Run the converter
if __name__ == "__main__":
    main()